In [ ]:
#텐서플로우 기반 경사하강법 - ver2.X
# tensorflow로 직접 경사하강법(gradient descent)구현한 
# 기본적인 선형회귀 코드

#y = wx + b 이 직선을 데이터에 가장 잘 맞게 긋는
# 최적의 w,b를 찾아내는 것!
# 그래서 이 선으로 새로운 데이터가 들어왔을 때 잘 예측(추론)이 목표

import tensorflow as tf#tensorflow import

#데이터 준비: 2차원 리스트 [x,y]원소로 갖는 데이터값
data = [[2,81],[4,93],[6,91],[8,97]]
#입력(x),정답(y) 데이터 분리 - 리스트 컴프리헨션
x_train = [x_row[0] for x_row in data]#[2,4,6,8]
y_train = [y_row[1] for y_row in data]#[81,93,91,97]

#y = wx + b 에서의 가중치, 편향 생성 - 처음엔 랜덤한 아무선으로 긋는다
w = tf.Variable(tf.random.uniform([1],0,10,dtype=tf.float64, seed=0))
b = tf.Variable(tf.random.uniform([1],0,100,dtype=tf.float64, seed=0))
#tf.Variable(): 학습하면서 값이 변경될 변수
#랜덤한 선으로 시작 -> 오차 계산 -> 업데이트 -> 반복
#dtype=tf.float64 텐서플로우가 미분하기 편하게 실수로 해줌
#seed = 0 랜덤 시드 고정(재현성 확보)

#weight와 bias을 통해 계산되는 예상 Y값
def hypothesis(w,b):#예측값
    return x_train * w + b #y=ax+b
#x_train * w 이거 백터 연산 됨
# w가 tf.Variable tensor 객체라서
# 내부적으로 x_train이 tensor로 자동 변환 발생
# 원래는 명시적으로 x_train을 tf.constant쓰던지 numpy써야 맞다고

def cost(w,b):#손실계산함수(RMSE): 학습시에는 원래 루트안씌워도 됨 MSE써야 더 좋음
    # 루트씌움 평균(예측값 - 실제값)^2
    return tf.sqrt(tf.reduce_mean(tf.square(hypothesis(w,b) - y_train))) 

#oprimizer 생성: SGD 확률적 경사하강법 사용, 파라미터 최적화 알고리즘
#이 계산식으로 gradient, learning 이용해서 파라미터(w,b)업데이트 함
opt = tf.keras.optimizers.SGD(learning_rate=0.1)
#learning_rate너무 크면 발산, 폭주, 너무 작으면 학습 오래걸림
#새로운 w = 기존 w - (기울기 * 학습률)
for i in range(2000): #학습 반복(epoch)
    #자동 미분을 위한 테이프
    #GradientTape(): 텐서플로우가 어떤 연산을 했는지 기록함 
    with tf.GradientTape() as tape:
        loss = cost(w,b)#기록할 내용 w,b
    # w를 기준으로 loss를 편미분한값, b를 기준으로 loss를 편미분한 값이 반환
    gradients = tape.gradient(loss,[w,b])#손실에 대한 w,b의 미분값 계산
    opt.apply_gradients(zip(gradients,[w,b]))#미분값으로 w와 b를 업데이트
    if i % 100 == 0:#100번마다 출력해서 찍어봄
        print(i, f'{loss.numpy()},{w.numpy()},{b.numpy()}')
        #loss라고 찍으면 텐서라서
        #.numpy() 값만 찍음

#tf.GradientTape()
#gradient 구하려면 미분해야함
#텐서플로우가 연산을 기록함
# 그리고 역으로 체인 룰 적용해서 자동 수행(역전파, backpropagation)
# 자동 미분용 연산 추적 시스템이라고,,

#zip(gradients,[w,b])
#[(gradient_w, w),(gradient_b, b)]
#옵티마이저가 원하는 형태 (gradient, variable)
#그래야 w,b 각각 업데이트

#gradients = tape.gradient(loss,[w,b])
#loss를 기준으로 w,b각각 얼마나 수정해야하는가? 계산함
#gradient = loss(손실함수)를 각 파라미터로 편미분한 값

#opt.apply_gradients(zip(gradients,[w,b]))
#진짜 학습이 일어나는 부분
#zip() 

#tf.Variable(): tensorfolw의 학습 가능한 상태값 객체
# 값 저장 + gradient 추적 + 값 수정 가능을 지원하는
# Tensor객체
# 텐서 플로우는 w = tf.Variable(3.0)
# 이런식으로 만들면 tensor 저장, GPU 올릴 수 있음
# 자동 미분 가능, 옵티마이저가 수정 가능 상태가 됨
# tf.Variable은 학습 대상 파라미터 변수라고 보면 된다..

#tf.random.uniform(): 랜덤 생성 함수
# 균등분포(nuiform, 모든 값이 같은 확률) 기반 랜덤값생성
#tf.random.uniform(shape,minval=0,maxval=None,dtype=tf.float32,seed=None)
#tf.random.uniform([1],0,10)
#[1]이건 1차원 원소 1개 tensor, 최소값 0, 최댓값 10
# 0~10사이의 랜덤값을 균등한 확률로 1차원 원소 1개 텐서로 반환
#shape에 아규먼트 []넣으면 스칼라(숫자1개)
#[1] 원소 1개 백터(1차원) 
#[5] 원소 5개 백터(1차원)
#[2,3] 2행 3열 행렬(2차원)

#학습흐름
# 1. 랜덤 w,b 시작
# 2. 예측값 계산
# 3. 손실 계산
# 4. gradient 계산
# 5. w,b 수정
# 6. 반복
# 7. 손실 최소화

#기대값 y = 2.3x + 79

# 1100 3.6305540630073287,[3.2027198],[73.61295429]
# 1200 3.188492302009098,[2.85788392],[75.67078735]
# 1300 2.9913781951450655,[2.62869122],[77.03851139]
# 1400 2.91808407393797,[2.48934855],[77.87004881]
# 1500 2.8931334108850093,[2.40815297],[78.35458935]
# 1600 2.88492240852091,[2.36159555],[78.63242411]
# 1700 2.8822515242146736,[2.33504637],[78.79085827]
# 1800 2.881386068602266,[2.3199343],[78.88104064]
# 1900 2.8811059828737577,[2.31133744],[78.93234303]